<a href="https://colab.research.google.com/github/AnthonyChenEE/robotics-coding/blob/main/robotic_arm_control_pid.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
class RoboticArm:
  def __init__(self):
    self.position = None # current robotic arm position
    self.gripper_open = True # gripper is open or not

  # 动作原语：移动到指定位置

In [ ]:
def move_to_position(self, new_position):
  print(f"Moving to {new_position}")
  self.position = new_position

# 动作原语:执行抓取动作
def grab(self):
  if self.gripper_open:
    print("Grabbing the object")
    self.gripper_open = False

# 动作原语：执行释放动作
def release(self):
  if not self.gripper_open:
    print("Releasing the object")
    self.gripper_open = True

# 动作级规划：组合动作原语来执行复杂任务
def execute_task(self, start_pos, end_pos):
  self.move_to_position(start_pos) # 移动到初始位置
  self.grab() # 抓取物体
  self.move_to_position(end_pos) # 移动到目标位置
  self.release() # 放置物体

In [ ]:
block_names = detect_objects("blocks")
bowl_names = detect_objects("bowls")
for bowl_name in bowl_names:
  if is_empty (bowl_name):
    empty_bowl = bowl_name
    break
objs_to_stack = [empty_bowl] + block_names
stack_objects(objs_to_stack)

In [ ]:
def is_empty(name):

def stack_objects(obj_names):
  n_objs = len(obj_names)
  for i in range(n_objs - 1):
    obj0 = obj_names[i + 1]
    obj1 = obj_names[i]
    pick_place(obj0, obj1)

In [ ]:
  # 動作API
  set_velocity()
  pick_obj(name)
  voice_out()

In [ ]:
objects = ['yellow block', 'green block', 'yellow bowl', 'blue block', 'blue bowl', 'green bowl']
# the yellow block on the yellow bowl
say('OK-putting the yellow block on the yellow bowl')
put_first_on_second ('yellow block', 'yellow bowl')
objects = ['yellow block', 'green block', 'yellow bowl','blue block', 'blue bowl', 'green bowl']
#which block did you move.
say('I moved the yellow block')
objects = ['yellow block', 'green block', 'yellow bowl', 'blue block', 'blue bowl', 'green bowl']
# move the green block to the top right corner/
say('Go it-putting')

In [ ]:
import numpy as np
from scipy.optimize import curve_fit
# define model function
def model (S, a, b, c):
  return a * (S ** (-b)) + c
# actual measurement data
actual_distances = np.array([5, 10, 15, 20, 25, 30, 35, 40, 45, 50, 55, 60])
image_areas = np.array([81, 33, 21, 16, 13, 11, 9, 8, 7, 6, 6, 5])
# use curve_fit
popt, _ = curve_fit(model, image_areas, actual_distances)
# output fit parameters
a, b, c = popt
print(f"Fit Parameters a = {a}, b = {b}, c = {c}")

Fit Parameters a = 274.24401084265855, b = 0.9257357933429033, c = -0.4662578881914762


In [ ]:
def Image_Processing(self, img):
  '''
  形態學transformation去除細小的干擾因素
  :param img: 輸入初始圖像
  :return: 檢測的輪廓點集
  '''
  # 將圖像->灰度圖
  gray_img = cv.cvtColor(img, cv.COLOR_RGB2GRAY)
  # 獲取不同形狀的結構元素，create a 5 * 5 matrix structuring element, for形態學操作
  kernel = cv.getStructuringElement(cv.MORPH_RECT, (5, 5))
  # 形態學閉操作，對灰度圖像，以去除小干擾
  dst_img = cv.morphologyEx(gray_img, cv.MORPH_CLOSE, kernel)
  # image二值化操作，對processed images，閾值 = 10，generate二值images
  ret, binary = cv.threshold(dst_img, 10, 225, cv.THRESH_BINARY)
  # 獲取輪廓point set(coordinates)，search for二值圖像中的contour
  find_contours = cv.findContours(binary, cv.RETR_EXTERNAL, cv.CHAIN_APPROX_SIMPLE)
  # 根據returned contour num，選擇合適的contour list
  if len(find_contours) == 3: contours = find_contours[1]
  else: contours = find_contours[0]
  # return detected contour point set
  return contours

In [ ]:
def get_detect_area(self, img):
  '''
  object detection - 獲取目標區域
  '''
  try:
    # 1. corret image pre-processing
    img_copy = image.copy()

    # 2. use LoadImages to pre-process
    img_copy, ratio, pad = self.letterbox(img_copy, new_shape=640, stride=32) # stride: 跨步
    # HWC->CHW, BGR->RGB
    img_copy = img_copy.transpose((2, 0, 1))[:-1]
    img_copy = np.ascontiguosarray(img_copy)

    # 3. convert to tnesor and ensure data types matched
    img_tensor = torch.from_numpy(img_copy).to(self.device)
    img_tensor = img_tensor.half() if self.half else img_tensor.float # convert by data tupes
    img_tensor /= 255.0
    if img_tensor.ndimension() == 3:
      img_tensor = img_tensor.unsqueeze(0)

    # add debug information
    print(f"Input tensor type: {img_tensor.dtype}")
    print(f"Model weight type: {next(self.model.parameters()).dtype}")

    # 4. reasoning
    with torch.no_grad():
      pred = self.model(img_tensor, argument=False) [0]

    # 5. NMS processing non-maximum supression
    pred = non_max_supression(
        pred,
        conf_thres = 0.25,
        iou_thres = 0.45,
        classes = None,
        agnostic = False
    )

    # 6. process detection results
    msg = {}
    for i, det in enumerate(pred): # detection results of every images
      if det is not None and len(det): # ensure det is not None and det is not empty
        # 縮放coordinates: img_size->original image size
        det[:, :4] = scale_coords(img_tensor.shape[2], det[:, :4], img.shape).round()
        # process every detection results
        for *xyxy, conf, cls in reversed(det):
          #  convert to integer coordinates
          bbox = [int(x) for x in xyxy]
          x1, y1, x2, y2 = bbox

          # calculate sizes of areas and types
          area = (x2 - x1) * (y2 - y1)
          name = self.names[int(cls)]

          # draw boundary frames and labels
          label = f'{name} {conf:2f}'
          plot_one_box(xyxy, img, label=label, color=self.colors[int(cls)], line_thickness=2)

          # draw central points
          cx = (x1 + x2) // 2
          cy = (y1 + y2) // 2
          cv. circle(img, (cx, cy), 5, (0, 0, 225), -1)

          # store results
          if area > 300: # area size threshold
            msg[name] = float(area) # only store area threshold
            print(f"detected {name}: area size = {area}")
    return img, msg

  except Exception as e:
    logging.error(f"Detection error: {str(e)}")
    import traceback
    traceback.print_exc()
    return img, None

In [2]:
class IncrementalPID:
  def __init__(self, P, I, D, target=0.0):
    # PID control parameters
    self.Kp = P
    self.Ki = I
    self.Kd = D

    # controller status variables
    self.PID_Output = 0.0 # PID controller output
    self.Target_Value = target # system target value

    # output 限制parameters
    self.Output_Max = 0 # upper bound of output
    self.Output_Min = 0 # lower bound of output
    self.Limit_Output = False # whether opens 輸出限制 or not

    # 誤差recordings
    self.Error = 0.0 # 偏差
    self.LastError = 0.0 # last 誤差
    self.LastLastError = 0.0 # last last 誤差

  # set PID controller parameters
  def calculate(self, nowValue):
    # calculate current 誤差
    self.Error = nowValue - self.Target_Value

    # calculate 增量 value
    # Δu(k) = Kp[e(k) - e(k-1)] + Ki*e(k) + Kd[e(k) - 2e(k-1) + e(k-2)]
    IncrementValue = self.Kp * (self.Error - self.LastError) +\
    self.Ki * self.Error +\
    self.Kd *(self.Error - 2 * self.LastError + self.LastLastError)

    # update output value
    self.PID_Output += IncrementValue
    # update 誤差 recordings
    self.LastLastError = self.LastError # keep last last error
    self.LastError = self.Error # keep last error

    # output 限制
    if self.Limit_Output and self.PID_Output > self.Output_Max:
      self.PID_Output = self.Output_Max
    if self.Limit_Output and self.PID_Output < self.Output_Min:
      self.PID_Output = self.Output_Min
    return self.PID_Output

  def set_target(self, target):
    """set target values"""
    self.Target_Value = target

  def set_limit_output(self, min, max):
    if min == 0 and max == 0:
      # turn off output 限制
      self.Limit_Output = False
      self.Output_Min = 0
      self.Output_Max = 0
    else:
      # open output 限制
      self.Limit_Output = True
      self.Output_Min = min
      self.Output_Max = max

  def set_pid_param(self, P, I, D):
    """reset PID parameters and controller status (clear output and 誤差 recordings)"""
    # update PID parameters
    self.Kp = P
    self.Ki = I
    self.Kd = D

    # reset controller status
    self.PID_Output = 0 # clear output
    self.Error = 0.0 # clear current 誤差
    self.LastError = 0.0 # clear last 誤差
    self.LastLastLastError = 0.0 # clear last last 誤差